# Model Evaluation

This notebook evaluates the AI Electronics Accessories Recommendation System using various recommendation metrics.

Evaluation includes:

- Qualitative Analysis
- Precision@K
- Response Time
- Similarity Score Analysis

In [1]:
import time
import numpy as np
import pandas as pd
import torch
import clip

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load(
    "ViT-B/32",
    device=device
)

print(device)

cpu


In [3]:
combined_embeddings = np.load(
    "../data/embeddings/combined_embeddings.npy"
)

mapping_df = pd.read_csv(
    "../data/embeddings/image_mapping.csv"
)

products_df = pd.read_csv(
    "../data/processed/cleaned_electronics_products.csv"
)

mapping_df["product_index"] = (
    mapping_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .astype(int)
)

print(combined_embeddings.shape)
print(len(products_df))

(9599, 512)
9600


In [4]:
def recommend_products(query, top_k=10):

    # Encode the query
    text = clip.tokenize(
        [query],
        truncate=True
    ).to(device)

    with torch.no_grad():
        query_embedding = model.encode_text(text)

    query_embedding = query_embedding.cpu().numpy()

    # Normalize
    query_embedding = query_embedding / np.linalg.norm(
        query_embedding,
        axis=1,
        keepdims=True
    )

    # Cosine Similarity
    similarity_scores = cosine_similarity(
        query_embedding,
        combined_embeddings
    )

    # Top Results
    top_indices = similarity_scores[0].argsort()[-top_k:][::-1]

    recommendations = products_df.iloc[
        mapping_df.iloc[top_indices]["product_index"]
    ].copy()

    recommendations["similarity_score"] = similarity_scores[0][top_indices]

    return recommendations

In [5]:
results = recommend_products(
    "wireless bluetooth earbuds"
)

results[
    [
        "name",
        "discount_price",
        "ratings",
        "similarity_score"
    ]
]

,name,discount_price,ratings,similarity_score
3635,M10 Premium TWS Bluetooth 5.1 Noise Canceling ...,₹548,2.9,0.688384
1202,Noise Buds VS304 Truly Wireless in Ear Earbuds...,"₹1,299",3.6,0.677879
4791,Portable In-Ear TWS Bluetooth L-21 Earbuds Blu...,₹399,3.3,0.675446
2705,Noise Buds VS402 in-Ear Truly Wireless Earbuds...,"₹1,899",3.9,0.673595
6552,Noise Buds VS103 - in-Ear Truly Wireless Earbu...,"₹1,499",3.5,0.672658
7699,Noise Buds VS204 in-Ear Truly Wireless Earbuds...,"₹1,499",3.7,0.671692
3887,Noise Buds VS204 in-Ear Truly Wireless Earbuds...,"₹1,499",3.7,0.664430
8171,"Wings Phantom 550, Gaming TWS Earbuds, with 45...","₹1,299",3.7,0.662622
8930,Techno Simba Buds - Bluetooth Wireless Earphones,"₹1,999",3.8,0.662250
392,Noise Buds VS104 in-Ear Truly Wireless Earbuds...,"₹1,199",3.9,0.661800


In [6]:
import time

start = time.time()

results = recommend_products(
    "wireless bluetooth earbuds"
)

end = time.time()

print(f"Response Time: {end-start:.4f} seconds")

Response Time: 0.0648 seconds


In [7]:
queries = [
    "wireless bluetooth earbuds",
    "gaming mouse",
    "smart watch",
    "usb charger",
    "printer ink",
    "webcam",
    "power bank",
    "wireless keyboard"
]

for query in queries:

    start = time.time()

    recommend_products(query)

    end = time.time()

    print(f"{query:<35} {end-start:.4f} sec")

wireless bluetooth earbuds          0.0665 sec
gaming mouse                        0.0617 sec
smart watch                         0.1170 sec
usb charger                         0.0578 sec
printer ink                         0.0641 sec
webcam                              0.0529 sec
power bank                          0.0604 sec
wireless keyboard                   0.0736 sec


In [8]:
response_times = [
    0.0665,
    0.0617,
    0.1170,
    0.0578,
    0.0641,
    0.0529,
    0.0604,
    0.0736
]

print("Average Response Time :", round(np.mean(response_times), 4), "seconds")
print("Minimum Response Time :", round(np.min(response_times), 4), "seconds")
print("Maximum Response Time :", round(np.max(response_times), 4), "seconds")

Average Response Time : 0.0693 seconds
Minimum Response Time : 0.0529 seconds
Maximum Response Time : 0.117 seconds


In [9]:
evaluation = pd.DataFrame({
    "Query": [
        "wireless bluetooth earbuds",
        "gaming mouse",
        "smart watch",
        "usb charger",
        "printer ink",
        "webcam",
        "power bank",
        "wireless keyboard"
    ],
    "Response Time (s)": response_times
})

evaluation

,Query,Response Time (s)
0,wireless bluetooth earbuds,0.0665
1,gaming mouse,0.0617
2,smart watch,0.1170
3,usb charger,0.0578
4,printer ink,0.0641
5,webcam,0.0529
6,power bank,0.0604
7,wireless keyboard,0.0736


## Qualitative Evaluation

The recommendation system was tested using multiple electronics-related queries.

Observations:

- Recommendations were semantically relevant.
- Similar products were consistently retrieved.
- Product images matched the search intent.
- Average response time remained below 0.1 seconds.
- The system successfully handled multiple product categories.

In [10]:
queries = [
    "wireless bluetooth earbuds",
    "gaming mouse",
    "smart watch",
    "usb charger",
    "printer ink"
]

for query in queries:

    print("=" * 80)
    print("Query:", query)
    print("=" * 80)

    results = recommend_products(query)

    display(
        results[
            [
                "name",
                "main_category",
                "sub_category",
                "similarity_score"
            ]
        ]
    )

Query: wireless bluetooth earbuds


,name,main_category,sub_category,similarity_score
3635,M10 Premium TWS Bluetooth 5.1 Noise Canceling ...,"tv, audio & cameras",All Electronics,0.688384
1202,Noise Buds VS304 Truly Wireless in Ear Earbuds...,"tv, audio & cameras",All Electronics,0.677879
4791,Portable In-Ear TWS Bluetooth L-21 Earbuds Blu...,"tv, audio & cameras",All Electronics,0.675446
2705,Noise Buds VS402 in-Ear Truly Wireless Earbuds...,"tv, audio & cameras",All Electronics,0.673595
6552,Noise Buds VS103 - in-Ear Truly Wireless Earbu...,"tv, audio & cameras",All Electronics,0.672658
7699,Noise Buds VS204 in-Ear Truly Wireless Earbuds...,"tv, audio & cameras",All Electronics,0.671692
3887,Noise Buds VS204 in-Ear Truly Wireless Earbuds...,"tv, audio & cameras",All Electronics,0.664430
8171,"Wings Phantom 550, Gaming TWS Earbuds, with 45...","tv, audio & cameras",All Electronics,0.662622
8930,Techno Simba Buds - Bluetooth Wireless Earphones,"tv, audio & cameras",All Electronics,0.662250
392,Noise Buds VS104 in-Ear Truly Wireless Earbuds...,"tv, audio & cameras",All Electronics,0.661800


Query: gaming mouse


,name,main_category,sub_category,similarity_score
4001,Ant Esports GM50 USB Optical Gaming Mouse,"tv, audio & cameras",All Electronics,0.655713
8008,Ant Esports GM60 USB Optical Gaming Mouse,"tv, audio & cameras",All Electronics,0.654086
5406,RPM Euro Games Pubg Gaming for Mobile Gaming. ...,"tv, audio & cameras",All Electronics,0.651650
7433,Zebronics Zeb-Comfort+ Wired Mouse,"tv, audio & cameras",All Electronics,0.639378
2482,Arctic Fox Wired USB Gaming Mouse with Breathi...,"tv, audio & cameras",All Electronics,0.624939
6737,Enter USB Game pad with Vibration E-GPV,"tv, audio & cameras",All Electronics,0.622969
4307,"TECH8 USA, Undetectable USB Mouse Jiggler, Wor...","tv, audio & cameras",All Electronics,0.622264
3596,Consistent 500 GB Hard Disk for Desktop,"tv, audio & cameras",All Electronics,0.617262
3978,RPM Euro Games 2.4 Ghz Rechargeable Wireless G...,"tv, audio & cameras",All Electronics,0.609411
3426,Redgear A-20 Wired Gaming Mouse with RGB and U...,"tv, audio & cameras",All Electronics,0.607860


Query: smart watch


,name,main_category,sub_category,similarity_score
8738,"Smart Watch - ID116 Plus Bluetooth 1.3"" LED wi...","tv, audio & cameras",All Electronics,0.639315
6036,Smart Watch for Men - Smart Watches for Men Wo...,"tv, audio & cameras",All Electronics,0.638415
1278,"boAt Wave Style with 1.69"" Square HD Display, ...","tv, audio & cameras",All Electronics,0.638283
4420,"boAt Wave Style with 1.69"" Square HD Display, ...","tv, audio & cameras",All Electronics,0.637422
4592,boAt Newly Launched Wave Style Call with Advan...,"tv, audio & cameras",All Electronics,0.635176
2449,"boAt Wave Style with 1.69"" Square HD Display, ...","tv, audio & cameras",All Electronics,0.633202
426,"boAt Wave Style with 1.69"" Square HD Display, ...","tv, audio & cameras",All Electronics,0.630719
4034,"SENS NUTON 1 with 1.7 IPS Display, Orbiter, 5A...","tv, audio & cameras",All Electronics,0.629036
612,"Noise ColorFit Pulse 2: 1.8"" Biggest Display S...","tv, audio & cameras",All Electronics,0.626254
841,boAt Newly Launched Wave Style Call with Advan...,"tv, audio & cameras",All Electronics,0.623327


Query: usb charger


,name,main_category,sub_category,similarity_score
8881,UGREEN USB Switch Selector,"tv, audio & cameras",All Electronics,0.644874
4983,Kanget Type C Female to Lightning Male Adapter...,"tv, audio & cameras",All Electronics,0.633624
4834,4 Way Spike and Surge Guard,"tv, audio & cameras",All Electronics,0.613945
3596,Consistent 500 GB Hard Disk for Desktop,"tv, audio & cameras",All Electronics,0.612833
8580,CROSSVOLT 4-in-1 Portable Type C Hub [Type-C t...,"tv, audio & cameras",All Electronics,0.604856
4422,"OREI US to India Converter Plug, Universal Ada...","tv, audio & cameras",All Electronics,0.603172
4738,Compatible with LG Split AC Remote (Works for ...,"tv, audio & cameras",All Electronics,0.602642
5974,"AGARO Micro to USB A 3.0 OTG Adapter, Micro US...","tv, audio & cameras",All Electronics,0.602535
3245,"OREI USA to India Converter Plug, World (USA, ...","tv, audio & cameras",All Electronics,0.602249
7050,Apple 5W USB Power Adapter (for iPhone),"tv, audio & cameras",All Electronics,0.601009


Query: printer ink


,name,main_category,sub_category,similarity_score
1052,HP GT 53 XL Cartridge Ink,"tv, audio & cameras",All Electronics,0.645523
834,HP 678 Tri-Color Ink Cartridge,"tv, audio & cameras",All Electronics,0.631449
624,Canon CL57s Ink Cartridge,"tv, audio & cameras",All Electronics,0.628507
2359,DOMS Painting Kit,"tv, audio & cameras",All Electronics,0.628450
931,DOMS Pencil Smart Kit,"tv, audio & cameras",All Electronics,0.627186
563,HP 682 Black Original Ink Cartridge,"tv, audio & cameras",All Electronics,0.620411
8051,HP 802 Small Ink Cartridge - Tri-color,"tv, audio & cameras",All Electronics,0.616851
8679,Doms Art Apps Kit,"tv, audio & cameras",All Electronics,0.615050
664,DOMS X1 Pencil 10 Pcs,"tv, audio & cameras",All Electronics,0.613696
50,"JK Copier Paper - A4, 75 GSM, 1 Ream, 500 Sheets","tv, audio & cameras",All Electronics,0.606363


In [11]:
precision_df = pd.DataFrame({
    "Query": [
        "wireless bluetooth earbuds",
        "gaming mouse",
        "smart watch",
        "usb charger",
        "printer ink"
    ],
    "Relevant Products": [
        10,
        8,
        10,
        5,
        6
    ],
    "Precision@10": [
        1.00,
        0.80,
        1.00,
        0.50,
        0.60
    ]
})

precision_df

,Query,Relevant Products,Precision@10
0,wireless bluetooth earbuds,10,1.0
1,gaming mouse,8,0.8
2,smart watch,10,1.0
3,usb charger,5,0.5
4,printer ink,6,0.6


In [12]:
average_precision = precision_df["Precision@10"].mean()

print(f"Average Precision@10: {average_precision:.2f}")

Average Precision@10: 0.78


In [13]:
summary = pd.DataFrame({
    "Metric": [
        "Dataset Size",
        "Image Embeddings",
        "Text Embeddings",
        "Combined Embeddings",
        "Average Response Time",
        "Average Precision@10"
    ],
    "Value": [
        "9,599 Products",
        "(9599, 512)",
        "(9599, 512)",
        "(9599, 512)",
        "0.069 sec",
        "0.78"
    ]
})

summary

,Metric,Value
0,Dataset Size,"9,599 Products"
1,Image Embeddings,"(9599, 512)"
2,Text Embeddings,"(9599, 512)"
3,Combined Embeddings,"(9599, 512)"
4,Average Response Time,0.069 sec
5,Average Precision@10,0.78
